**Project Structure**

agrismart-ai/
│
├── app.py
├── config.py
├── requirements.txt
├── .env
├── README.md
│
├── data/
│   └── agriculture.pdf
│
├── pages/
│   ├── 1_Chatbot.py
│   ├── 2_PDF_Knowledge.py
│   ├── 3_Farm_Calculator.py
│   └── 4_About.py
│
├── tools/
│   ├── pdf_tool.py
│   ├── search_tool.py
│   ├── calculator.py
│   └── memory.py
│
├── rag/
│   ├── embeddings.py
│   ├── vectorstore.py
│   └── retriever.py
│
└── assets/

requirements.txt



In [44]:
# Install required packages
!pip install streamlit
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf
!pip install python-dotenv
!pip install duckduckgo-search
!pip install tiktoken
!pip install pandas
!pip install numpy
!pip install requests
!pip install langchain-huggingface

In [25]:
import os

# Set environment variables
os.environ['GROQ_API_KEY'] = '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'
os.environ['MODEL_NAME'] = 'llama-3.3-70b-versatile'
os.environ['TEMPERATURE'] = '0.3'
os.environ['MAX_TOKENS'] = '4096'

In [26]:
# config.py
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

MODEL_NAME = os.getenv(
    "MODEL_NAME",
    "llama-3.3-70b-versatile"
)

TEMPERATURE = float(
    os.getenv("TEMPERATURE", 0.3)
)

MAX_TOKENS = int(
    os.getenv("MAX_TOKENS", 4096)
)

Initialize the Groq LLM

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

In [14]:
# Conversation Memory
import streamlit as st

if "messages" not in st.session_state:
    st.session_state.messages = []

2026-07-11 17:42:57.890 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:42:57.891 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-07-11 17:42:57.894 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:42:57.894 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [15]:
# Chat Interface
import streamlit as st

st.title("🌱 AgriSmart AI")

prompt = st.chat_input("Ask me anything about farming...")

if prompt:

    st.session_state.messages.append(
        {
            "role":"user",
            "content":prompt
        }
    )

    response = llm.invoke(prompt)

    st.session_state.messages.append(
        {
            "role":"assistant",
            "content":response.content
        }
    )

for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])

2026-07-11 17:44:28.477 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.564 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-11 17:44:28.565 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.571 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:44:28.573 Thread 'MainThread': mi

In [16]:
# PDF Upload
uploaded_file = st.file_uploader(
    "Upload Agricultural PDF",
    type="pdf"
)

2026-07-11 17:45:21.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:45:21.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:45:21.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:45:21.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 17:45:21.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
# Create the 'data' directory and a dummy 'agriculture.pdf' file if they don't exist
!mkdir -p data
!touch data/agriculture.pdf

In [21]:
# Read PDF
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []  # Initialize documents as an empty list

# In a typical Streamlit app run via `streamlit run`, uploaded_file would contain the file data.
# However, when running cells individually, st.file_uploader won't correctly populate uploaded_file.
# We check for its existence in globals() but it will likely be None here.
if 'uploaded_file' in globals() and uploaded_file is not None:
    try:
        # To process an uploaded file, you would typically save its content to a temporary file
        # and then pass the path to PyPDFLoader. This part is commented out as it requires
        # a valid uploaded_file object which is not present in sequential cell execution.
        # with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
        #     tmp_file.write(uploaded_file.getvalue())
        #     temp_file_path = tmp_file.name
        # loader = PyPDFLoader(temp_file_path)
        # documents = loader.load()
        # os.remove(temp_file_path)
        print("Uploaded file detected, but direct processing requires running the full Streamlit app.")
    except Exception as e:
        print(f"Error processing uploaded file: {e}")
elif os.path.exists("data/agriculture.pdf") and os.path.getsize("data/agriculture.pdf") > 0:
    try:
        # Attempt to load the pre-existing file if it's not empty
        loader = PyPDFLoader("data/agriculture.pdf")
        documents = loader.load()
        print(f"Loaded {len(documents)} pages from data/agriculture.agriculture.pdf.")
    except Exception as e:
        print(f"Error loading data/agriculture.pdf: {e}")
else:
    print("No valid PDF uploaded or found at 'data/agriculture.pdf'. 'documents' list is empty.")
    print("Please upload a PDF via the Streamlit uploader when running the full app, \
           or ensure 'data/agriculture.pdf' contains a valid PDF file (not empty).")

No valid PDF uploaded or found at 'data/agriculture.pdf'. 'documents' list is empty.
Please upload a PDF via the Streamlit uploader when running the full app,            or ensure 'data/agriculture.pdf' contains a valid PDF file (not empty).


### Create a Mock Document for Testing

In [46]:
from langchain_core.documents import Document

mock_content = """
Agriculture is the backbone of many economies, providing food, fiber, and raw materials.
Sustainable agriculture practices are crucial for long-term ecological balance and food security.
These practices include crop rotation, organic farming, water conservation, and reduced pesticide use.
Precision agriculture, leveraging technologies like IoT and AI, is transforming farming by optimizing resource allocation and improving yields.
Key crops include wheat, corn, rice, and soybeans, which are staple foods globally.
Livestock farming also plays a significant role, contributing to dairy, meat, and wool production.
Challenges in agriculture include climate change, soil degradation, and pest resistance, necessitating continuous innovation and research.
"""

# Check if documents is empty before adding mock data
if not documents:
    mock_doc = Document(
        page_content=mock_content,
        metadata={'source': 'mock_document.txt', 'page': 1}
    )
    documents.append(mock_doc)
    print("Mock document created and added to the 'documents' list for testing.")
else:
    print("Documents list is not empty, skipping mock document creation.")

print(f"Current number of documents: {len(documents)}")

Mock document created and added to the 'documents' list for testing.
Current number of documents: 1


In [29]:
from google.colab import files
import shutil
import os

# Ensure the 'data' directory exists
!mkdir -p data

print("Please upload your agriculture.pdf file:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

    # Move the uploaded file to the desired path
    destination_path = 'data/agriculture.pdf'
    if os.path.exists(destination_path):
        os.remove(destination_path) # Remove existing dummy file if present
    shutil.move(fn, destination_path)
    print(f"File moved to {destination_path}")

# Verify that the file exists and is not empty
if os.path.exists('data/agriculture.pdf') and os.path.getsize('data/agriculture.pdf') > 0:
    print("agriculture.pdf successfully uploaded and moved to the 'data' directory.")
else:
    print("Error: agriculture.pdf is either missing or empty after upload.")

Please upload your agriculture.pdf file:


Error: agriculture.pdf is either missing or empty after upload.


In [ ]:
# Verify file existence and size after upload
import os

file_path = 'data/agriculture.pdf'

if os.path.exists(file_path):
    file_size = os.path.getsize(file_path)
    if file_size > 0:
        print(f"'agriculture.pdf' exists in 'data/' and has a size of {file_size} bytes. This is good!")
    else:
        print(f"'agriculture.pdf' exists but is empty. Please ensure you uploaded a valid PDF with content.")
else:
    print(f"'agriculture.pdf' does not exist in 'data/'. The upload was likely unsuccessful.")

!ls -lh data/

In [50]:
# Split Documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print(f"Created {len(docs)} document chunks.")

Created 1 document chunks.


In [45]:
# Create Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [48]:
from langchain_community.vectorstores import FAISS

if len(docs) == 0:
    raise ValueError("No document chunks were created. Please ensure 'data/agriculture.pdf' exists and contains content.")

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [51]:
# Save the Vector Store (or create a new cell for this if loading is intended later)
vectorstore.save_local(
    folder_path="faiss_index",
    index_name="index"
)

In [53]:
import streamlit as st

# Retrieve Relevant Information
retriever = vectorstore.as_retriever()

# Define a default prompt for testing in the notebook environment
# In a full Streamlit app, this would be set by st.chat_input
if 'prompt' not in locals() or prompt is None:
    prompt = "What are sustainable agriculture practices?"

results = retriever.invoke(prompt)
print(f"Retrieved documents for prompt: '{prompt}'")
for i, doc in enumerate(results):
    print(f"Document {i+1}:\n{doc.page_content[:200]}...\n")

Retrieved documents for prompt: 'What are sustainable agriculture practices?'
Document 1:
Agriculture is the backbone of many economies, providing food, fiber, and raw materials. 
Sustainable agriculture practices are crucial for long-term ecological balance and food security. 
These pract...



In [55]:
# Web Search Tool
!pip install -U ddgs
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

answer = search.run(
    "Current maize price in Nigeria"
)

print(answer)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.9 MB/s eta 0:00:00
Jul 17, 2025 ... Before now, a 100kg bag of maize cost N85,000 to N90,000 the market, but the price has now dropped to N50,000 to N55,000. A farmer, Harisu Abdullahi Maresu, ... Jun 9, 2026 ... Maize Prices in Nigeria ... As of July 2026, the maize price in Nigeria available is US$ 0.28 per kg (131 NGN), down 73.7% year-on-year. Nigeria exported 18 ... Mar 23, 2026 ... This upward trend in export prices reflects a broader expansion over the period. Conversely, the average import price for maize in Nigeria decreased by 6.9% in ... Sep 24, 2025 ... This morning at Killa Market in Ogun State, I observed people trading maize and conducted a market survey with buyers and sellers. Today, a sack of maize is ... Feb 25, 2026 ... Why Food Prices Keep Rising in Nigeria 2026. Food inflation has become one of the most pressing economic realit

In [56]:
# Farm Profit Calculator
def calculate_profit(cost, revenue):

    return revenue - cost

Tool Routing

In [58]:
from langchain_groq import ChatGroq

# Re-initialize the LLM to ensure it uses the correct model name
# This addresses potential inconsistencies from earlier runs or state changes.
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

# Tool Routing
if "price" in prompt.lower():

    response = search.run(prompt)

elif "profit" in prompt.lower():

    response = calculate_profit(
        500000,
        850000
    )

else:

    response = llm.invoke(prompt).content

In [59]:
# Streamlit Sidebar
with st.sidebar:

    st.title("🌾 AgriSmart AI")

    st.markdown("---")

    st.write("✔ AI Chat")

    st.write("✔ PDF Knowledge")

    st.write("✔ Calculator")

    st.write("✔ Web Search")

2026-07-11 18:45:25.752 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.756 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.757 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.758 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.759 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-11 18:45:25.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Save Dependencies and Model

In [60]:
# Generate requirements.txt
!pip freeze > requirements.txt

print("requirements.txt created successfully.")
print("FAISS vector store already saved in 'faiss_index/' directory.")

requirements.txt created successfully.
FAISS vector store already saved in 'faiss_index/' directory.
